In [ ]:
# ---------------- Imports ----------------
import os
import json
from collections import defaultdict
import yaml


In [ ]:
# ---------------- Args ----------------
DATASET_TO_BALANCE = "20260115T095923-combined-claims-full"

FRAMINGS = {
    "original",
    "authoritative",
    "consensus",
    "emotional",
    "prestige",
    "sensationalist",
}


# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATA_FOLDER = os.path.join(PROJ_STORE, "data", "augmented-processed")


INPUT_ROOT = os.path.join(DATA_FOLDER, DATASET_TO_BALANCE)
OUTPUT_ROOT = os.path.join(f"{INPUT_ROOT}-balanced")
os.makedirs(OUTPUT_ROOT, exist_ok=True)




In [ ]:

# -------- Helpers --------
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

def get_base_id(claim_id):
    return claim_id.split(":")[0]

# -------- Main --------
for split in ["train", "dev", "test"]:
    input_dir = os.path.join(INPUT_ROOT, split)
    output_dir = os.path.join(OUTPUT_ROOT, split)

    if not os.path.exists(input_dir):
        continue

    for fname in os.listdir(input_dir):
        if not fname.endswith(".jsonl"):
            continue

        input_path = os.path.join(input_dir, fname)
        output_path = os.path.join(output_dir, fname)

        # group rows by base claim
        groups = defaultdict(list)

        for row in load_jsonl(input_path):
            base_id = get_base_id(row["claim_id"])
            groups[base_id].append(row)

        # filter complete sets
        kept_rows = []

        for base_id, rows in groups.items():
            framings_present = {r["framing_type"] for r in rows}

            if FRAMINGS.issubset(framings_present):
                # keep only one per framing (in case of duplicates)
                seen = set()
                for r in rows:
                    ft = r["framing_type"]
                    if ft in FRAMINGS and ft not in seen:
                        kept_rows.append(r)
                        seen.add(ft)

        write_jsonl(output_path, kept_rows)

        print(f"{split}/{fname}: kept {len(kept_rows)} rows")

